In [1]:

# ============================================================
# BRAIN TUMOR CLASSIFICATION — INCEPTIONV3 TRANSFER LEARNING
# Complete Kaggle pipeline in one cell
# ============================================================

import os
import random
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

!nvidia-smi


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, models
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

# -------------------- 1. CONFIGURATION -----------------------

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS_FEATURE_EXTRACTION = 30
EPOCHS_FINE_TUNING = 10
VALID_EXT = {".jpeg", ".jpg", ".bmp", ".png"}

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


DATA_DIR = Path("/kaggle/input/datasets/anaselkhadiri2964/brain-scan/Data")

if DATA_DIR is None:
    candidates = []

    for root in Path("/kaggle/input").rglob("*"):
        if not root.is_dir():
            continue

        subdirs = [p for p in root.iterdir() if p.is_dir()]
        valid_classes = [
            d for d in subdirs
            if any(
                f.is_file() and f.suffix.lower() in VALID_EXT
                for f in d.iterdir()
            )
        ]

        if len(valid_classes) >= 2:
            candidates.append((root, len(valid_classes)))

    if not candidates:
        raise FileNotFoundError(
            "Dataset not found. Set DATA_DIR to your dataset root."
        )

    # Prefer the candidate containing the most class directories.
    DATA_DIR = max(candidates, key=lambda item: item[1])[0]

DATA_DIR = Path(DATA_DIR)
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Dataset directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

# -------------------- 2. LOAD IMAGE PATHS -------------------

class_names = sorted([
    d.name for d in DATA_DIR.iterdir()
    if d.is_dir()
    and any(
        f.is_file() and f.suffix.lower() in VALID_EXT
        for f in d.iterdir()
    )
])

if len(class_names) != NUM_CLASSES:
    raise ValueError(
        f"Expected {NUM_CLASSES} classes, found {len(class_names)}: "
        f"{class_names}. Check DATA_DIR."
    )

image_paths = []
labels = []

for class_name in class_names:
    class_dir = DATA_DIR / class_name

    files = sorted([
        f for f in class_dir.iterdir()
        if f.is_file() and f.suffix.lower() in VALID_EXT
    ])

    for path in files:
        image_paths.append(str(path))
        labels.append(class_name)

print("\nClasses:", class_names)
print("Number of classes:", len(class_names))
print("Total images:", len(image_paths))

for name, count in sorted(Counter(labels).items()):
    print(f"{name}: {count} images")

if len(image_paths) == 0:
    raise ValueError("No supported images found.")

# -------------------- 3. CLASS DISTRIBUTION -----------------

plt.figure(figsize=(9, 4))
sns.countplot(x=labels, order=class_names)
plt.title("Brain MRI Dataset Distribution")
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

# -------------------- 4. VISUALIZE SAMPLE IMAGES ------------

fig, axes = plt.subplots(
    len(class_names), 3, figsize=(10, 3 * len(class_names))
)
axes = np.asarray(axes).reshape(len(class_names), 3)

for i, class_name in enumerate(class_names):
    class_paths = [
        p for p, label in zip(image_paths, labels)
        if label == class_name
    ]

    for j, path in enumerate(class_paths[:3]):
        image = cv2.imread(path)

        if image is not None:
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, IMG_SIZE)
            axes[i, j].imshow(image)
            axes[i, j].set_title(class_name)
        else:
            axes[i, j].set_title("Could not load image")

        axes[i, j].axis("off")

plt.tight_layout()
plt.show()

# -------------------- 5. ENCODE LABELS ----------------------

le = LabelEncoder()
y_encoded = le.fit_transform(labels)
NUM_CLASSES = len(le.classes_)
print("Label mapping:", dict(enumerate(le.classes_)))

# -------------------- 6. TRAIN / VAL / TEST SPLIT ------------

paths = np.array(image_paths)
y = np.array(y_encoded)

# 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    paths,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print("\nDataset split:")
print(f"Train: {len(X_train)} ({len(X_train)/len(paths):.1%})")
print(f"Val:   {len(X_val)} ({len(X_val)/len(paths):.1%})")
print(f"Test:  {len(X_test)} ({len(X_test)/len(paths):.1%})")

# -------------------- 7. BALANCE TRAINING DATA ---------------

# Oversample minority classes by duplicating image paths.
# Random augmentation is applied separately each time an image
# is loaded, so duplicated paths can produce different images.

train_counts = Counter(y_train)
max_count = max(train_counts.values())

balanced_paths = list(X_train)
balanced_labels = list(y_train)

for class_idx in range(NUM_CLASSES):
    class_paths = X_train[y_train == class_idx]
    count = len(class_paths)
    diff = max_count - count

    if diff > 0:
        chosen = np.random.choice(
            class_paths, size=diff, replace=True
        )
        balanced_paths.extend(chosen.tolist())
        balanced_labels.extend([class_idx] * diff)

balanced_paths = np.array(balanced_paths)
balanced_labels = np.array(balanced_labels)

# Shuffle balanced training data
permutation = np.random.permutation(len(balanced_paths))
balanced_paths = balanced_paths[permutation]
balanced_labels = balanced_labels[permutation]

print("\nBalanced training distribution:")
for idx, name in enumerate(le.classes_):
    print(f"{name}: {np.sum(balanced_labels == idx)}")

# -------------------- 8. TF.DATA PIPELINE -------------------

AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(
        image_bytes, channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    image = preprocess_input(image)
    return image, tf.cast(label, tf.int32)


def augment_image(image, label):
    # Conservative augmentation for MRI images.
    # No vertical flips or strong geometric transformations.
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.08)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)

    # Small noise addition after InceptionV3 preprocessing.
    noise = tf.random.normal(
        tf.shape(image), mean=0.0, stddev=0.02
    )
    image = image + noise

    return image, label


def make_dataset(image_paths, image_labels, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(
        (image_paths, image_labels)
    )

    if training:
        dataset = dataset.shuffle(
            buffer_size=len(image_paths),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_image, num_parallel_calls=AUTOTUNE
    )

    if training:
        dataset = dataset.map(
            augment_image, num_parallel_calls=AUTOTUNE
        )

    dataset = dataset.map(
        lambda image, label: (
            image, tf.one_hot(label, NUM_CLASSES)
        ),
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(BATCH_SIZE)
    return dataset.prefetch(AUTOTUNE)


train_ds = make_dataset(
    balanced_paths, balanced_labels, training=True
)
val_ds = make_dataset(X_val, y_val)
test_ds = make_dataset(X_test, y_test)


# -------------------- 9. BUILD INCEPTIONV3 ------------------

gpus = tf.config.list_physical_devices("GPU")
print("Detected GPUs:", len(gpus))

if len(gpus) == 0:
    raise RuntimeError(
        "No GPU detected. Enable GPU in Kaggle Settings."
    )

strategy = tf.distribute.MirroredStrategy()

print("Number of replicas:", strategy.num_replicas_in_sync)

with strategy.scope():

    base_model = InceptionV3(
        weights="imagenet",
        include_top=False,
        input_shape=(*IMG_SIZE, 3)
    )

    base_model.trainable = False

    model = models.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-4
        ),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

model.summary()


# -------------------- 10. FEATURE EXTRACTION ----------------

checkpoint_path = OUTPUT_DIR / "best_brain_tumor_model.keras"

callbacks = [
    ModelCheckpoint(
        str(checkpoint_path),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FEATURE_EXTRACTION,
    callbacks=callbacks
)

# -------------------- 11. PLOT FIRST TRAINING CURVES --------

def plot_history(history, title):
    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(history.history["loss"], label="Train Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.title(f"{title} - Loss")
    plt.xlabel("Epoch")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history["accuracy"], label="Train Accuracy")
    plt.plot(
        history.history["val_accuracy"],
        label="Validation Accuracy"
    )
    plt.title(f"{title} - Accuracy")
    plt.xlabel("Epoch")
    plt.legend()

    plt.tight_layout()
    plt.show()


plot_history(history, "Feature Extraction")


# -------------------- 12. FINE-TUNING ------------------------

# Unfreeze InceptionV3
base_model.trainable = True

# Freeze all layers first
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze only the last 100 layers, keeping BatchNorm frozen
for layer in base_model.layers[-100:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

# IMPORTANT: Recompile inside the SAME MirroredStrategy scope
with strategy.scope():
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-5
        ),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

# Fine-tune
history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE_TUNING,
    callbacks=callbacks
)

plot_history(history_ft, "Fine-Tuning")


# -------------------- 13. TEST EVALUATION -------------------

# Reload the best validation-loss checkpoint.
model = tf.keras.models.load_model(str(checkpoint_path))

test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Generate predictions in the same order as X_test.
y_prob = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=np.arange(NUM_CLASSES),
    target_names=le.classes_,
    zero_division=0
))

cm = confusion_matrix(
    y_test, y_pred, labels=np.arange(NUM_CLASSES)
)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.title("Test Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.tight_layout()
plt.show()

# -------------------- 14. SAVE FINAL MODEL ------------------

final_model_path = OUTPUT_DIR / "googlenet_brain_tumor.keras"
model.save(str(final_model_path))

# Save class names for future inference.
class_names_path = OUTPUT_DIR / "brain_tumor_class_names.txt"
class_names_path.write_text(
    "\n".join(le.classes_), encoding="utf-8"
)

print("\nTraining pipeline completed.")
print("Best checkpoint:", checkpoint_path)
print("Final model:", final_model_path)
print("Class names:", class_names_path)
print("Kaggle output directory:", OUTPUT_DIR)


KeyboardInterrupt: 